# 📝 과제 02: 사내 PT 준비 팀

## 만들 결과

가상 SaaS 회사 **누리소프트** 사업기획팀의 발표 준비 도우미를 만듭니다. 발표 요청을 받으면 필요한 자료(매출, 고객 설문, 경쟁사 요금제, 작성 원칙)만 골라 동시에 조사하고, 조사 메모의 수치만 써서 발표 구성안을 작성합니다. 검수자가 수치와 출처를 조사 메모와 대조하고, 통과하지 못하면 라운드 상한까지만 다시 쓰거나 다시 조사한 뒤 사람에게 넘깁니다.

- 요청에 필요한 자료원만 조사 작업으로 나누고 작업 수만큼 조사원을 동시에 실행합니다.
- 병렬 조사 메모를 작업 ID별로 합쳐 발표 구성안과 3분 요약 스크립트를 쓰고, 검수에서 반려되면 다시 쓰거나 다시 조사합니다.
- 팀 회의 3분 공유와 임원회의 10분 제안을 같은 그래프로 실행해 작업 수가 다른지 확인합니다.

교안 02의 동적 계획, `Send` 병렬 실행, 리듀서, 핸드오프, 검수 루프를 사내 발표 준비에 적용합니다. 자료 조회 도구, 프롬프트, 결과 모으기·작성·검수 노드는 제공하고, 리듀서·라우팅 함수·계획과 조사 노드·핸드오프 도구·다음 단계 결정 노드·그래프 연결을 직접 작성합니다.

<img src="images/assignment02_overview.png" width="1050" alt="planner가 작업을 나눠 researcher를 작업 수만큼 동시에 실행하고, collect가 메모를 모아 writer가 발표 구성안을 씁니다. writer는 핸드오프로 reviewer나 supervisor로 넘기고, supervisor가 다시 쓰기, 다시 조사, 종료를 정합니다.">

## 준비: 데이터와 제공 코드

- **자료:** `data/assignment02_*.json` 4개. 분기별 매출(36행), 고객 설문(응답 342명), 경쟁사 요금제(6개), PT 작성 원칙(7조항)입니다(수업용 가상 자료).
- **순서:** 리듀서 -> 라우팅 함수 -> 계획 노드 -> 조사 노드 -> 핸드오프 도구 -> 다음 단계 결정 노드 -> 그래프 연결과 실행 -> 서술
- **제공:** 환경 설정, 자료 조회 도구, 프롬프트, 결과 모으기·발표 구성안 작성·검수 노드
- **작성:** `[작성]` 셀의 `...` 자리에 번호 주석대로 코드를 넣고, 바로 아래 자가채점 셀을 실행하세요. 자가채점은 모델을 호출하지 않습니다. 7번만 모델을 실제로 호출합니다.

In [ ]:
# 라이브러리, 모델, 파일 경로를 준비합니다.
import os
import json
import re
from datetime import date
from pathlib import Path
from typing import Annotated, Literal
from typing_extensions import TypedDict
from dotenv import find_dotenv, load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, Send
from pydantic import BaseModel, Field
from IPython.display import Image, Markdown

# material_dir는 data·output이 있는 단원 폴더입니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 공통 키는 저장소 최상위 .env에서 읽습니다. 이미 읽은 키는 덮어쓰지 않습니다.
load_dotenv(material_dir.resolve().parent / ".env")
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
llm = ChatOpenAI(model=model_name, use_responses_api=True)
print("답변 모델:", model_name)

In [ ]:
# 자가채점에서 모델 대신 고정 응답을 끼울 때 쓰는 도구입니다. 자가채점 셀 안에서만 씁니다.
from copy import deepcopy
from types import SimpleNamespace
from unittest.mock import Mock, patch

In [ ]:
# 입력 자료는 data에서 읽고, 결과 파일은 output에 저장합니다.
def read_json(name):
    """data 폴더의 JSON 파일을 읽어 파이썬 객체로 반환합니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """결과를 output 폴더의 JSON 파일로 저장합니다."""
    (output_dir / name).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")


# 매출 데이터: 분기, 지역, 고객군별 매출, 유효 계약 수, 해지 수
sales = read_json("assignment02_sales.json")
# 설문 데이터: 고객군별 응답 수, 가격 민감도, 요청 기능, NPS
survey = read_json("assignment02_survey.json")
# 경쟁사 요금 데이터: 요금, 최소 인원, 기능, 출처와 확인일
competitors = read_json("assignment02_competitors.json")
# 작성 원칙 데이터: 발표 대상별 조항
guideline = read_json("assignment02_guideline.json")
print("매출", len(sales["rows"]), "행 | 설문 응답", sum(survey["respondents"].values()), "명 | 경쟁사 요금제", len(competitors["plans"]), "개 | 작성 원칙", len(guideline["clauses"]), "조항")

In [ ]:
# [보조 함수] 매출 합산: 매출 도구가 분기와 묶음 기준으로 합계를 낼 때 씁니다.
def sum_sales(quarter, group_key, group_name):
    """분기와 묶음 기준(region 또는 segment)으로 매출, 유효 계약 수, 해지 수를 합산합니다. group_name이 빈 문자열이면 전체입니다."""
    revenue = 0
    contracts = 0
    churned = 0
    for row in sales["rows"]:
        if row["quarter"] == quarter and (group_name == "" or row[group_key] == group_name):
            revenue += row["revenue"]
            contracts += row["contracts"]
            churned += row["churned"]
    return revenue, contracts, churned


# [도구] 매출 조회: 증감률과 해지율은 도구가 계산해 돌려줍니다.
@tool
def get_sales_summary(quarter: str, group_by: Literal["지역", "고객군"] = "지역") -> str:
    """분기('2026Q3' 모양)의 매출 요약을 지역별 또는 고객군별로 반환합니다. 매출(백만 원), 전분기·전년 동기 대비 증감률, 유효 계약 수, 해지 수와 해지율을 계산해 줍니다."""
    if group_by == "지역":
        group_key = "region"
        names = ["수도권", "영남", "호남", "충청"]
    else:
        group_key = "segment"
        names = ["대기업", "중견기업", "중소기업"]

    compare = sales["compare"].get(quarter, {})
    lines = [f"[{sales['source_id']}] {quarter} {group_by}별 매출 요약 (매출 단위: 백만 원)"]
    for name in names + [""]:
        revenue, contracts, churned = sum_sales(quarter, group_key, name)
        if revenue == 0:
            return f"{quarter} 매출 자료가 없습니다."
        # 증감률은 도구가 계산합니다. 비교 분기 자료가 있을 때만 붙입니다.
        trend = []
        for label, other in compare.items():
            if other is not None:
                before = sum_sales(other, group_key, name)[0]
                trend.append(f"{label} 대비 {(revenue - before) * 100 / before:+.1f}%")
        trend_text = ""
        if trend:
            trend_text = " (" + ", ".join(trend) + ")"
        label = name
        if name == "":
            label = "전체"
        lines.append(f"{label}: 매출 {revenue:,}{trend_text}, 유효 계약 {contracts:,}건, 해지 {churned:,}건 (해지율 {churned * 100 / (contracts + churned):.1f}%)")
    return "\n".join(lines)


# [도구] 설문 조회: 응답 수를 비율로 바꿔 돌려줍니다.
@tool
def get_survey_stats(topic: Literal["가격 민감도", "요청 기능", "NPS"], segment: Literal["중소기업", "중견기업"] = "중소기업") -> str:
    """2026년 9월 고객 설문에서 한 주제의 응답 수와 비율을 고객군별로 계산해 반환합니다. 가격 민감도는 사용자 10명 팀 기준 월 구독료 지불 의향입니다."""
    total = survey["respondents"][segment]
    head = f"[{survey['source_id']}] {survey['title']} {segment} (응답 {total}명)"
    if topic == "NPS":
        counts = survey["nps"][segment]
        score = (counts["promoters"] - counts["detractors"]) * 100 / total
        return f"{head} NPS: 추천 {counts['promoters']}명, 중립 {counts['passives']}명, 비추천 {counts['detractors']}명, NPS {score:+.1f}"
    if topic == "가격 민감도":
        counts = survey["price"][segment]
        head += " 월 구독료 지불 의향(사용자 10명 팀 기준)"
    else:
        counts = survey["features"][segment]
        head += " 요청 기능(복수 응답, 응답자 대비 비율)"
    lines = [head]
    for name, count in counts.items():
        lines.append(f"- {name}: {count}명 ({count * 100 / total:.1f}%)")
    return "\n".join(lines)


# [도구] 경쟁사 요금 조회: 10명 기준 월 비용을 계산하고 오래된 자료는 표시합니다.
@tool
def get_competitor_pricing(segment: Literal["중소기업", "중견기업"] = "중소기업") -> str:
    """고객군의 경쟁사·자사 요금제를 비교해 반환합니다. 사용자 10명 기준 월 비용(최소 인원 적용, 부가세 별도), 기능, 출처와 확인일을 주고, 기준일에서 90일이 지난 자료는 재확인 필요로 표시합니다."""
    as_of = date.fromisoformat(competitors["as_of"])
    lines = [f"[경쟁사 요금 비교] {segment} 기준일 {as_of} (사용자 10명 기준, 부가세 별도)"]
    for plan in competitors["plans"]:
        if plan["segment"] != segment:
            continue
        # 최소 인원이 10명보다 크면 최소 인원만큼 청구합니다.
        billed = 10
        if plan["min_users"] > 10:
            billed = plan["min_users"]
        age = (as_of - date.fromisoformat(plan["checked_on"])).days
        notice = ""
        if age > 90:
            notice = f", 확인 후 {age}일 경과(재확인 필요)"
        lines.append(f"{plan['cp_id']} {plan['company']} {plan['plan']}: 사용자당 월 {plan['price']:,}원, 최소 {plan['min_users']}명, "
                     f"10명 기준 월 {plan['price'] * billed:,}원(청구 {billed}명), 기능 {', '.join(plan['features'])}, "
                     f"출처 {plan['source']} 확인일 {plan['checked_on']}{notice}")
    return "\n".join(lines)


# [도구] 작성 원칙 조회: 발표 대상에 해당하는 조항만 돌려줍니다.
@tool
def get_pt_guideline(audience: Literal["임원회의", "팀회의"]) -> str:
    """사내 PT 작성 원칙에서 발표 대상에 해당하는 조항(조항 ID와 본문)을 반환합니다."""
    lines = [f"[{guideline['source_id']}] {audience} 발표 원칙"]
    for clause in guideline["clauses"]:
        if audience in clause["audiences"]:
            lines.append(f"{clause['clause_id']}: {clause['text']}")
    return "\n".join(lines)

In [ ]:
# 조사원 설정: 자료원 이름으로 그 작업이 쓸 도구를 찾습니다. 조사원은 자기 작업의 도구 하나만 받습니다.
TOOLS_BY_SOURCE = {
    "sales": [get_sales_summary],
    "survey": [get_survey_stats],
    "competitor": [get_competitor_pricing],
    "guideline": [get_pt_guideline],
}

researcher_system = ("사내 사업기획팀 조사원입니다. 지시문에서 분기, 고객군, 주제를 찾아 주어진 도구를 호출하고, 도구 결과를 짧게 정리하세요. "
                     "출처 ID, 수치, 날짜는 도구 결과 그대로 옮기고, 직접 계산한 값과 도구 결과에 없는 내용은 쓰지 마세요.")

In [ ]:
# 3분기 고객군별 매출 요약을 조회해 봅니다. 도구에는 인자 이름을 키로 한 딕셔너리를 넘깁니다.
print(get_sales_summary.invoke({"quarter": "2026Q3", "group_by": "고객군"}))

## 1. 병렬 조사 메모를 작업 ID별로 합치세요

`merge_findings`는 **리듀서**입니다. 노드가 `findings`를 반환할 때마다 기존 값과 새 값을 합칩니다. 노드와 달리 State가 아니라 값 두 개(`old`, `new`)를 받습니다.

<img src="images/node_focus/assignment02_researcher.png" width="1050" alt="과제 전체 흐름에서 researcher 노드만 강조합니다.">

**작성:** `merge_findings(old, new)` 리듀서를 설계하세요.

**목적:** 조사원 여러 명이 같은 필드에 동시에 메모를 씁니다. 같은 작업을 다시 조사하면 옛 메모를 새 메모로 바꾸고, 다른 작업의 메모는 남깁니다.

**구현할 처리**

1. 기존 메모 `old`를 복사한 새 dict를 만듭니다.
2. 새 메모 `new`를 얹어 같은 키는 새 값으로 바꿉니다.
3. 합친 dict를 반환합니다. `old`와 `new`는 직접 바꾸지 않습니다.

**반환값: `dict`**

합친 새 dict입니다. 키는 작업 ID, 값은 조사 메모 문자열입니다.

**반환 예시:** `merge_findings({"sales": "옛 메모", "survey": "설문 메모"}, {"sales": "새 메모"})`

```python
{"sales": "새 메모", "survey": "설문 메모"}
```

In [ ]:
# [리듀서] 기존 조사 메모에 새 메모를 합칩니다.
def merge_findings(old, new):
    """기존 메모 old에 새 메모 new를 합친 새 dict를 반환합니다. 같은 키는 새 값으로 바꿉니다."""
    # 1. 기존 메모 old를 복사한 새 dict를 만드세요.
    ...

    # 2. 새 메모 new를 얹어 같은 키는 새 값으로 바꾸세요.
    ...

    # 3. 합친 dict를 반환하세요.
    ...

In [ ]:
# [자가채점] 같은 키 교체, 겹치지 않는 키 유지, 입력 불변을 확인합니다. 모델을 호출하지 않습니다.
def grade_question_1():
    old = {"sales": "옛 매출 메모", "survey": "옛 설문 메모"}
    new = {"sales": "새 매출 메모", "competitor": "경쟁사 메모"}
    old_before = dict(old)
    new_before = dict(new)
    merged = merge_findings(old, new)
    assert isinstance(merged, dict), "합친 결과는 dict여야 합니다. return을 확인하세요."
    assert merged == {"sales": "새 매출 메모", "survey": "옛 설문 메모", "competitor": "경쟁사 메모"}, "같은 키는 새 값으로 바꾸고, 겹치지 않는 키는 모두 남기세요."
    assert old == old_before and new == new_before, "old와 new를 직접 바꾸지 말고 새 dict를 반환하세요."
    assert merge_findings({}, new) == new, "기존 메모가 비어 있어도 동작해야 합니다."
    print("리듀서 확인 완료")


grade_question_1()

## 2. 작업마다 조사원을 하나씩 띄우세요

`assign_research`는 노드가 아닌 **라우팅 함수**입니다. 반환한 `Send`의 개수만큼 `researcher`가 동시에 실행됩니다.

<img src="images/node_focus/assignment02_planner.png" width="1050" alt="과제 전체 흐름에서 planner 노드만 강조합니다.">

팀 State와 보조 함수는 제공합니다. 아래 코드를 실행한 뒤 사용하세요.

| 제공 코드 | 입력과 반환값 |
|---|---|
| `PTState` | 팀 전체가 공유하는 State(`TypedDict`). `findings` 필드에 1번의 `merge_findings`가 쓰입니다. |
| `make_input(request, max_rounds=2)` | 요청 문장으로 시작 State dict를 만듭니다. `tasks`는 `[]`, `round`는 0입니다. |
| `format_findings(findings)` | 작업 ID별 메모를 `[작업 ID]` 제목을 붙인 한 문자열로 묶습니다. |

In [ ]:
# [State] PT 준비 팀 그래프의 공유 데이터
class PTState(TypedDict):
    # 처음 설정
    request: str  # 발표 준비 요청 원문
    max_rounds: int  # 검수 반려 뒤 다시 쓰기·다시 조사를 맡길 수 있는 최대 횟수

    # planner·researcher·collect
    tasks: list[dict]  # planner: 이번에 할 조사 작업
    findings: Annotated[dict[str, str], merge_findings]  # researcher: 작업 ID별 조사 메모
    brief: str  # collect: 조사 메모를 작업 ID별로 묶은 글 (교안의 기획 메모와 달리 종합하지 않음)

    # writer·reviewer·supervisor
    draft: str  # writer: 에이전트가 쓴 발표 구성안. request_review 도구가 draft 인자로 받아 여기에 저장합니다
    review: dict  # reviewer·request_research: 통과 여부·문제 종류·의견
    feedback: str  # supervisor: 다음 라운드에 전할 검수 의견 (planner는 반영 후 비움)
    round: int  # supervisor: 반려되어 writer나 planner로 돌려보낸 횟수
    status: str  # supervisor: 최종 상태('통과' 또는 '라운드 상한 도달: 사람 확인 필요')


# [보조 함수] 그래프 시작 입력 만들기: 요청 하나로 모든 필드의 초깃값을 채웁니다.
def make_input(request, max_rounds=2):
    """요청 하나로 시작 상태를 만듭니다. 결과 필드는 비우고 라운드는 0부터 셉니다."""
    return {
        "request": request,
        "max_rounds": max_rounds,
        "tasks": [],
        "findings": {},
        "brief": "",
        "draft": "",
        "review": {},
        "feedback": "",
        "round": 0,
        "status": "진행 중",
    }


# [보조 함수] LLM 컨텍스트 만들기: researcher가 모은 조사 결과(findings)를 writer·reviewer 프롬프트에 넣을 문자열로 바꿉니다.
def format_findings(findings):
    """작업 ID와 조사 메모를 한 문자열로 묶습니다."""
    blocks = []
    for task_id, text in findings.items():
        # 작업 ID를 대괄호로 붙여 어느 조사에서 나온 메모인지 구분하게 합니다.
        blocks.append(f"[{task_id}]\n{text}")
    return "\n\n".join(blocks)

**작성:** `assign_research(state)` 라우팅 함수를 설계하세요.

**목적:** planner가 만든 작업마다 조사원 실행 하나씩을 예약합니다.

**구현할 처리**

1. 빈 목록을 만듭니다.
2. `state["tasks"]`의 작업을 하나씩 돌며 `Send`를 목록에 추가합니다. 첫 인자는 노드 이름 `"researcher"`, 둘째 인자는 `{"task": 작업 하나}`입니다. State 전체는 넘기지 않습니다.
3. 목록을 반환합니다. 작업이 없으면 빈 목록입니다.

**반환값: `list`**

작업 하나에 `Send` 하나가 대응하는 목록입니다.

**반환 예시:** `tasks`가 작업 한 개인 경우

```python
[Send("researcher", {"task": {"task_id": "sales", "source": "sales", "query": "2026년 3분기 고객군별 매출 조회"}})]
```

In [ ]:
# [라우팅 함수] 작업마다 researcher로 가는 Send를 만듭니다.
def assign_research(state: PTState):
    """작업 하나마다 researcher로 가는 Send를 만들어 목록으로 반환합니다."""
    # 1. Send를 담을 빈 목록을 만드세요.
    ...

    # 2. 작업을 하나씩 돌며 작업마다 Send 하나를 목록에 추가하세요.
    ...

    # 3. 목록을 반환하세요.
    ...

In [ ]:
# [자가채점] Send의 노드 이름과 전달값, 빈 작업 처리를 확인합니다. 모델을 호출하지 않습니다.
def grade_question_2():
    tasks = [{"task_id": "sales", "source": "sales", "query": "2026년 3분기 고객군별 매출 조회"},
             {"task_id": "guideline", "source": "guideline", "query": "임원회의 작성 원칙 조회"}]
    state = make_input("평가용 요청")
    state["tasks"] = tasks
    before = deepcopy(state)
    sends = assign_research(state)
    assert isinstance(sends, list), "Send 객체를 담은 목록을 반환하세요."
    assert len(sends) == 2, "작업 수만큼 Send를 만드세요."
    for send, task in zip(sends, tasks):
        assert isinstance(send, Send), "목록의 원소는 Send(...)여야 합니다."
        assert send.node == "researcher", "Send의 첫 인자는 노드 이름 'researcher'입니다."
        assert send.arg == {"task": task}, "Send에는 State 전체가 아니라 {'task': 작업 하나}만 담으세요."
    assert state == before, "입력 State를 직접 바꾸지 마세요."
    state["tasks"] = []
    assert assign_research(state) == [], "작업이 없으면 빈 목록을 반환합니다."
    print("라우팅 함수 확인 완료")


grade_question_2()

## 3. 요청을 읽고 조사 작업을 나누세요

`planner`는 요청에서 필요한 조사를 고르는 노드입니다. 요청마다 작업 수가 달라집니다.

<img src="images/node_focus/assignment02_planner.png" width="1050" alt="과제 전체 흐름에서 planner 노드만 강조합니다.">

자료원은 `sales`, `survey`, `competitor`, `guideline` 네 가지입니다.

| 제공 코드 | 입력과 반환값 |
|---|---|
| `planner_chain.invoke({"request": ..., "feedback": ...})` | 요청 문장과 검수 의견 문자열을 받아 `ResearchPlan`을 반환합니다. `plan.tasks`의 원소는 `source`, `query` 속성을 가집니다. |

In [ ]:
# 작업 하나는 자료원 하나이고, 작업 수는 planner가 요청을 읽고 정합니다.
class Task(BaseModel):
    source: Literal["sales", "survey", "competitor", "guideline"] = Field(description="이 작업이 쓰는 자료원 하나")
    query: str = Field(description="조사원에게 줄 한국어 지시 한 문장. 분기, 고객군, 주제가 있으면 넣는다")


class ResearchPlan(BaseModel):
    tasks: list[Task] = Field(description="필요한 조사 작업 1~4개")


planner_prompt = ChatPromptTemplate.from_messages([
    ("system", "사내 발표(PT) 준비 요청을 읽고 발표 구성안에 필요한 조사만 작업으로 나누세요. 작업 하나는 자료원 하나만 씁니다. "
     "sales=분기 매출·증감률·해지율(지역별 또는 고객군별), survey=고객 설문(가격 민감도, 요청 기능, NPS), "
     "competitor=경쟁사 요금제 비교(고객군별), guideline=사내 PT 작성 원칙(임원회의 또는 팀회의). 현재는 2026년 3분기입니다. "
     "요청이 묻지 않은 항목은 만들지 마세요. 임원회의에 제안하는 요청이면 guideline 작업도 포함하세요. 자료원마다 작업은 하나입니다. 작업은 1~4개입니다. "
     "검수 의견이 '없음'이 아니면 요청 전체가 아니라 그 의견에 적힌 빈 칸을 메우는 작업만 만드세요. "
     "같은 자료원을 다시 조사하는 작업의 새 메모는 옛 메모를 대신하므로, 지시문에 원래 요청이 필요로 한 분기, 고객군, 주제를 빠뜨리지 말고 모두 적으세요."),
    ("human", "요청: {request}\n검수 의견: {feedback}"),
])
planner_chain = planner_prompt | llm.with_structured_output(ResearchPlan)

**작성:** `planner(state)` 노드를 설계하세요.

**목적:** 요청과 검수 의견으로 이번에 할 조사 작업을 정합니다. 작업 ID는 모델이 아니라 코드가 만듭니다.

**구현할 처리**

1. `planner_chain`을 한 번 호출합니다. 입력은 `state["request"]`와 검수 의견이며, `state["feedback"]`이 비어 있으면 검수 의견으로 `"없음"`을 넘깁니다.
2. `plan.tasks`의 작업을 하나씩 `model_dump()`로 dict로 바꾸고 `task_id`에 `source` 값을 그대로 담습니다. 이미 나온 `source`는 건너뛰어 먼저 나온 작업만 남깁니다(최대 4개). ID가 자료원 이름이므로 다시 조사한 메모는 1번 리듀서가 옛 메모와 바꿉니다.
3. 작업 목록과 빈 `feedback`을 담은 dict를 반환합니다. 입력 State는 바꾸지 않습니다.

**반환값: `dict`**

| 키 | 반환할 값 |
|---|---|
| `tasks` | `source`, `query`, `task_id`를 가진 작업 dict의 목록 |
| `feedback` | 반영한 검수 의견을 비우는 빈 문자열 `''` |

**반환 예시:** 모델이 `sales`를 두 번, `guideline`을 한 번 말한 경우

```python
{
    "tasks": [{"source": "sales", "query": "2026년 3분기 고객군별 매출 조회", "task_id": "sales"},
              {"source": "guideline", "query": "임원회의 작성 원칙 조회", "task_id": "guideline"}],
    "feedback": ""
}
```

In [ ]:
# [노드] 요청과 검수 의견으로 조사 작업을 나눕니다.
def planner(state: PTState):
    """요청과 검수 의견을 읽고 이번에 할 조사 작업 목록을 반환합니다."""
    # 1. 검수 의견이 없으면 "없음"으로 바꿔 planner_chain을 한 번 호출하세요.
    ...

    # 2. 작업을 dict로 바꾸고 자료원 이름을 task_id로 붙이되, 이미 나온 자료원은 건너뛰세요.
    ...

    # 3. 작업 목록과 빈 feedback을 담은 dict를 반환하세요.
    ...

In [ ]:
# [자가채점] 고정 계획으로 task_id, 중복 제거, 검수 의견 처리를 확인합니다. 모델을 호출하지 않습니다.
def grade_question_3():
    plan = ResearchPlan(tasks=[Task(source="sales", query="2026년 3분기 고객군별 매출 조회"),
                               Task(source="sales", query="지역별 매출 다시 조회"),
                               Task(source="guideline", query="임원회의 작성 원칙 조회")])
    fake_chain = Mock()
    fake_chain.invoke.return_value = plan
    state = make_input("평가용 요청")
    before = deepcopy(state)
    with patch.dict(planner.__globals__, {"planner_chain": fake_chain}):
        first = planner(state)
    assert isinstance(first, dict) and "tasks" in first and "feedback" in first, "{'tasks': ..., 'feedback': ''} 모양의 dict를 반환하세요."
    assert fake_chain.invoke.call_count == 1, "planner_chain은 한 번만 호출하세요."
    sent = fake_chain.invoke.call_args.args[0]
    assert sent["request"] == "평가용 요청", "체인에 요청 원문을 넘기세요."
    assert sent["feedback"] == "없음", "첫 계획에는 검수 의견 대신 '없음'을 넘기세요."
    task_ids = []
    for task in first["tasks"]:
        task_ids.append(task["task_id"])
    assert task_ids == ["sales", "guideline"], "작업 ID는 자료원 이름이고, 같은 자료원은 먼저 나온 작업 하나만 남깁니다."
    assert first["tasks"][0]["query"] == "2026년 3분기 고객군별 매출 조회", "같은 자료원이 겹치면 먼저 나온 작업을 남기세요."
    assert first["tasks"][1]["source"] == "guideline", "task.model_dump()의 source, query를 그대로 보존하세요."
    assert first["feedback"] == "", "반영한 검수 의견은 빈 문자열로 비우세요."
    assert state == before, "입력 State를 직접 바꾸지 마세요."
    state["feedback"] = "경쟁사 조사가 없습니다."
    with patch.dict(planner.__globals__, {"planner_chain": fake_chain}):
        planner(state)
    assert fake_chain.invoke.call_args.args[0]["feedback"] == "경쟁사 조사가 없습니다.", "검수 의견이 있으면 그대로 체인에 넘기세요."
    print("계획 노드 확인 완료")


grade_question_3()

## 4. 작업 하나를 맡은 조사원을 만드세요

`researcher`는 `Send`로 작업 하나만 받는 노드라서 매개변수가 `state`가 아닌 `payload`입니다.

<img src="images/node_focus/assignment02_researcher.png" width="1050" alt="과제 전체 흐름에서 researcher 노드만 강조합니다.">

| 제공 코드 | 입력과 반환값 |
|---|---|
| `TOOLS_BY_SOURCE` | 자료원 이름(`"sales"` 등)을 키로, 그 작업이 쓸 도구 목록을 값으로 가진 dict |
| `researcher_system` | 조사원 시스템 프롬프트 문자열 |

**작성:** `researcher(payload)` 노드를 설계하세요.

**목적:** 작업의 자료원 도구 하나만 가진 에이전트로 조사하고, 작업 ID를 키로 메모를 돌려줍니다. 다른 자료원을 조회하지 않게 하려는 것입니다.

**구현할 처리**

1. `payload["task"]`에서 작업을 꺼냅니다.
2. `create_agent`로 에이전트를 만듭니다. 첫 인자는 모델 `llm`, `tools=`에는 `TOOLS_BY_SOURCE[작업의 source]`, `system_prompt=`에는 `researcher_system`을 넘깁니다.
3. 작업의 `query`를 `HumanMessage`에 담아 에이전트를 한 번 `invoke`합니다. 입력은 `{"messages": [...]}` 모양입니다.
4. 결과 `result["messages"]`의 마지막 메시지 `.text`를 메모로 삼아 `{"findings": {작업 ID: 메모}}`로 반환합니다. 메시지 기록 전체는 반환하지 않습니다. `findings`는 리듀서 필드이므로 이번 작업의 새 메모만 반환하고, 기존 메모까지 담아 반환하지 않습니다.

**반환값: `dict`**

| 키 | 반환할 값 |
|---|---|
| `findings` | `{작업 ID: 조사 메모 문자열}` 한 쌍으로 된 dict |

**반환 예시:** `task_id`가 `"guideline"`인 경우

```python
{"findings": {"guideline": "[GL-2026] 임원회의 발표 원칙 G-1: 첫 장표에 결론과 요청 사항을 먼저 씁니다. ..."}}
```

In [ ]:
# [노드] 작업 하나를 맡은 조사원
def researcher(payload):
    """작업의 자료원 도구 하나만 가진 에이전트로 조사하고, 작업 ID를 키로 한 메모를 반환합니다."""
    # 1. payload에서 작업을 꺼내세요.
    ...

    # 2. 작업의 자료원 도구만 담은 에이전트를 만드세요.
    ...

    # 3. 작업의 query를 사람 메시지로 에이전트에 한 번 실행하세요.
    ...

    # 4. 마지막 메시지의 text를 작업 ID를 키로 한 findings로 반환하세요.
    ...

In [ ]:
# [자가채점] 고정 에이전트로 도구 선택, 메시지 전달, 반환 키를 확인합니다. 모델을 호출하지 않습니다.
def grade_question_4():
    fake_agent = Mock()
    # content는 블록 목록이라 text와 다릅니다. 마지막 메시지의 text를 써야 통과합니다.
    fake_message = SimpleNamespace(text="채점용 원칙 메모", content=[{"type": "text", "text": "블록 형태의 내용"}])
    fake_agent.invoke.return_value = {"messages": [fake_message]}
    fake_create = Mock(return_value=fake_agent)
    task = {"task_id": "guideline", "source": "guideline", "query": "임원회의 작성 원칙을 조회하세요"}
    with patch.dict(researcher.__globals__, {"create_agent": fake_create}):
        result = researcher({"task": task})
    assert fake_create.call_count == 1, "에이전트는 작업마다 한 번 만듭니다."
    call = fake_create.call_args
    model_arg = call.kwargs.get("model")
    if call.args:
        model_arg = call.args[0]
    assert model_arg is llm, "create_agent의 첫 인자는 모델 llm입니다."
    tools_arg = call.kwargs.get("tools")
    if tools_arg is None and len(call.args) > 1:
        tools_arg = call.args[1]
    assert tools_arg == TOOLS_BY_SOURCE["guideline"], "tools에는 TOOLS_BY_SOURCE[작업의 source]를 넘기세요."
    assert call.kwargs.get("system_prompt") == researcher_system, "system_prompt에는 researcher_system을 넘기세요."
    assert fake_agent.invoke.call_count == 1, "에이전트는 한 번만 실행합니다."
    assert "임원회의 작성 원칙을 조회하세요" in str(fake_agent.invoke.call_args.args[0]), "작업의 query를 사람 메시지로 넘기세요."
    assert result == {"findings": {"guideline": "채점용 원칙 메모"}}, "{'findings': {작업 ID: 마지막 메시지의 text}}만 반환하세요."
    print("조사 노드 확인 완료")


grade_question_4()

## 5. 발표 구성안을 검수자에게 넘기는 핸드오프 도구를 만드세요

`request_review`는 노드가 아닌 **도구**입니다. writer 에이전트가 이 도구를 호출하면, 반환한 `Command`에 따라 바깥 그래프가 `reviewer` 노드로 이동합니다.

<img src="images/node_focus/assignment02_writer.png" width="1050" alt="과제 전체 흐름에서 writer 노드만 강조합니다.">

| 제공 코드 | 입력과 반환값 |
|---|---|
| `request_research(missing)` | 조사 메모가 부족하다고 알립니다. 바깥 그래프의 `supervisor`로 이동하는 `Command`를 반환합니다. |

**작성:** `request_review(draft)` 도구를 설계하세요. `@tool` 선언과 설명문은 제공됩니다.

**목적:** 완성한 발표 구성안을 바깥 State의 `draft`에 담아 `reviewer`로 넘깁니다.

**구현할 처리**

1. 이동할 노드(`goto`)를 `"reviewer"`로 정합니다.
2. 이동 범위(`graph`)를 에이전트 안이 아니라 바깥 그래프로 정합니다. 바깥 그래프를 가리키는 값은 `Command.PARENT`입니다.
3. 바깥 State에 반영할 값(`update`)으로 받은 발표 구성안만 `{"draft": ...}`로 담습니다. 위 세 값을 가진 `Command`를 반환합니다.

**반환값: `Command`**

| 인자 | 값 |
|---|---|
| `goto` | `"reviewer"` |
| `graph` | `Command.PARENT` |
| `update` | `{"draft": 받은 발표 구성안}` |

**반환 예시:** `request_review.invoke({"draft": "발표 구성안"})`

```python
Command(goto="reviewer", graph=Command.PARENT, update={"draft": "발표 구성안"})
```

In [ ]:
# [핸드오프 도구] 검수 요청: 완성한 발표 구성안을 reviewer에게 넘깁니다.
@tool
def request_review(draft: str) -> Command:
    """완성한 발표 구성안을 검수자(reviewer)에게 넘깁니다. draft에 발표 구성안 전문을 넣으세요."""
    # 이동할 노드(goto), 이동 범위(graph), 바깥 State에 쓸 값(update)을 담은 Command를 반환하세요.
    ...

In [ ]:
# [핸드오프 도구] 추가 조사 요청: 빠진 자료를 적어 supervisor에게 넘깁니다.
@tool
def request_research(missing: str) -> Command:
    """조사 메모만으로 발표 구성안을 쓸 수 없을 때 무엇이 빠졌는지 알리고 추가 조사를 요청합니다."""
    # planner로 바로 가지 않고 supervisor를 거쳐야 라운드 상한 검사를 받습니다. 빠진 내용은 review 필드에 검수 의견처럼 담습니다.
    review = {"passed": False, "issue": "research", "feedback": missing}
    return Command(goto="supervisor", graph=Command.PARENT, update={"review": review})

In [ ]:
# [자가채점] Command의 이동 노드, 이동 범위, 갱신값을 확인합니다. 모델을 호출하지 않습니다.
def grade_question_5():
    sent = request_review.invoke({"draft": "평가용 발표 구성안"})
    assert isinstance(sent, Command), "도구는 Command를 반환해야 합니다. return을 확인하세요."
    assert sent.goto == "reviewer", "goto는 노드 이름 'reviewer'입니다."
    assert sent.graph == Command.PARENT, "바깥 그래프로 이동하려면 graph=Command.PARENT가 필요합니다."
    assert sent.update == {"draft": "평가용 발표 구성안"}, "update에는 {'draft': 받은 발표 구성안}만 넣으세요."
    other = request_research.invoke({"missing": "경쟁사 조사가 없습니다."})
    assert other.goto == "supervisor" and other.graph == Command.PARENT, "제공된 request_research 셀은 고치지 마세요. supervisor로 이동해야 합니다."
    print("핸드오프 도구 확인 완료")


grade_question_5()

## 6. 검수 결과로 다음 단계를 정하세요

`supervisor`는 검수 판정과 라운드 수로 끝낼지, 다시 쓸지, 다시 조사할지 정하는 노드입니다.

<img src="images/node_focus/assignment02_supervisor.png" width="1050" alt="과제 전체 흐름에서 supervisor 노드만 강조합니다.">

| 제공 코드 | 입력과 반환값 |
|---|---|
| `writer(state)` | 조사 메모와 검수 의견으로 발표 구성안을 쓰는 에이전트 노드입니다. 5번 도구를 호출하면 바깥 그래프로 이동합니다. |
| `reviewer(state)` | LLM으로 검수하는 노드입니다. 발표 구성안을 조사 메모와 대조해 `{"review": {"passed": bool, "issue": "none" 또는 "rewrite" 또는 "research", "feedback": 문자열}}`를 반환합니다. |

In [ ]:
# writer는 두 핸드오프 도구를 가진 에이전트입니다. 도구를 부르면 바깥 그래프로 이동합니다.
writer_system = ("사내 사업기획팀 발표 담당자로서 '발표 구성안'과 '3분 요약 스크립트'를 한국어로 씁니다. "
                 "발표 구성안은 슬라이드마다 '제목, 핵심 메시지, 근거 수치(출처 ID)'를 적습니다. 요청이 묻지 않은 항목은 쓰지 마세요. "
                 "제출 전 체크리스트입니다. 검수자가 같은 기준으로 봅니다. "
                 "1. 수치는 조사 메모에 적힌 글자 그대로만 씁니다. 단위 변환, 반올림, 직접 계산(차이, 합계, 평균, 범위 만들기)을 하지 않습니다. "
                 "2. 구성안과 요약 스크립트의 모든 수치 바로 뒤에 그 수치의 출처 ID를 괄호로 붙입니다. 제목과 핵심 메시지 문장도 같습니다. "
                 "예: '625백만 원(SALES-FY26)', 여러 출처는 '월 90,000원(CP-A), 월 120,000원(CP-B)'처럼 수치마다 붙입니다. "
                 "3. 조사 메모가 말하지 않은 해석, 평가, 전망, 원인은 쓰지 않거나 문장 앞에 '(추정)'을 붙이고 근거 수치와 출처 ID를 적습니다. "
                 "4. 조사 메모에 작성 원칙(G-번호)이 있으면 지킵니다. 슬라이드 수 상한, 첫 장표의 결론과 요청 사항, 마지막 장표의 승인 사항, "
                 "'재확인 필요' 경쟁사 자료는 수치 없이 재확인 필요라고만 적기입니다. "
                 "승인 사항에 넣을 요금·일정이 조사 메모에 정해져 있지 않으면 지어내지 말고 '요금안(확정 전, 확인 필요)'처럼 항목만 적습니다. "
                 "5. 검수 의견으로 다시 쓸 때는 지적된 문장만 고치지 말고 전체를 1~4번으로 처음부터 다시 점검하세요. "
                 "요청이 물은 항목에 해당하는 조사 메모가 한 줄도 없을 때만 발표 구성안을 쓰지 말고 request_research로 무엇이 빠졌는지 알리세요. "
                 "다 쓰면 request_review 도구의 draft에 발표 구성안과 요약 스크립트 전문을 넣어 제출하세요.")

# 한 응답에 도구를 하나만 부르게 합니다. 핸드오프 두 개가 함께 나가면 한쪽만 반영됩니다.
writer_llm = ChatOpenAI(model=model_name, use_responses_api=True, model_kwargs={"parallel_tool_calls": False})

# writer 에이전트: 핸드오프 도구 두 개만 가집니다.
writer_agent = create_agent(writer_llm, tools=[request_review, request_research], system_prompt=writer_system)

In [ ]:
# [보조 함수] LLM 컨텍스트 만들기: writer 에이전트에 넣을 요청·조사 메모·검수 의견을 한 문자열로 묶습니다.
def writer_request(state):
    """요청, 조사 메모, 검수 의견을 writer에게 보낼 한 문자열로 묶습니다."""
    feedback = state["feedback"]
    # 첫 작성에는 검수 의견이 없으므로 "없음"을 넘깁니다.
    if not feedback:
        feedback = "없음"

    return f"요청: {state['request']}\n\n조사 메모:\n{state['brief']}\n\n검수 의견: {feedback}"


# [노드] 발표 구성안 작성
def writer(state: PTState):
    """조사 메모와 검수 의견으로 writer 에이전트를 실행합니다."""
    result = writer_agent.invoke({"messages": [HumanMessage(writer_request(state))]})

    # 핸드오프가 일어나면 아래 줄은 실행되지 않습니다.
    # 도구를 부르지 않고 끝났을 때만 마지막 답변을 발표 구성안으로 보고 reviewer에게 넘깁니다.
    return Command(goto="reviewer", update={"draft": result["messages"][-1].text})

In [ ]:
# 검수 판정 스키마: 통과 여부·문제 종류·의견
class Review(BaseModel):
    passed: bool = Field(description="고치지 않고 보고해도 되면 true")
    issue: Literal["none", "rewrite", "research"] = Field(
        description="통과면 none, 글 수정이면 rewrite, 조사 보완이면 research"
    )
    feedback: str = Field(
        description="반려 이유와 할 일 한두 문장. rewrite이면 고칠 문장을 따옴표로 인용하고, "
                    "research이면 다시 조사할 항목을 적는다. 통과(none)면 빈 문자열"
    )


reviewer_prompt = ChatPromptTemplate.from_messages([
    ("system", "발표 구성안과 3분 요약 스크립트를 조사 메모와 대조해 검수합니다. 아래 네 가지만 확인하고, 하나라도 어기면 issue=rewrite입니다. "
     "1. 수치가 조사 메모에 글자 그대로 있는가. 메모 수치로 직접 계산한 값(차이, 합계, 평균, 범위)은 위반입니다. "
     "2. 조사 메모에서 가져온 수치마다 출처 ID가 붙어 있는가. 괄호인지 대괄호인지 같은 표기 모양은 따지지 않습니다. "
     "요청에 나온 조건 숫자(발표 시간, 슬라이드 수, '3분기' 같은 기간, '10명 기준' 같은 기준 인원)는 출처 ID가 없어도 됩니다. "
     "3. 조사 메모가 말하지 않은 해석·전망·원인을 사실처럼 단정했는가. 문장 앞에 '(추정)'이 있으면 위반이 아닙니다. "
     "4. 조사 메모에 작성 원칙(G-번호)이 있으면 지켰는가. 작성 원칙 메모가 없는 요청에서는 슬라이드 수 같은 원칙을 따지지 않습니다. "
     "다음은 반려 사유가 아닙니다: 문장을 더 다듬을 여지, 조사 메모에 없는 내용(예: 메모에 없는 요금안이나 출시 일정)을 새로 넣으라는 요구, "
     "요청이 묻지 않은 항목이 빠진 것, '확인 필요'로 적어 둔 세부. "
     "요청이 물은 항목의 조사 메모 자체가 없어 구성안을 완성할 수 없으면 issue=research입니다. "
     "rewrite이면 feedback에 어긴 번호와 문제 문장을 따옴표로 인용합니다. 문제가 없으면 passed=true, issue=none입니다."),
    ("human", "요청: {request}\n\n조사 메모:\n{findings}\n\n발표 구성안:\n{draft}"),
])
reviewer_chain = reviewer_prompt | llm.with_structured_output(Review)


# [노드] 검수
def reviewer(state: PTState):
    """발표 구성안을 조사 메모와 대조한 검수 결과를 반환합니다."""
    result = reviewer_chain.invoke({"request": state["request"], "findings": format_findings(state["findings"]),
                                    "draft": state["draft"]})
    return {"review": result.model_dump()}

**작성:** `supervisor(state)` 노드를 설계하세요.

**목적:** 검수 판정과 라운드 수로 다음 단계를 정합니다. 반려된 초안을 통과한 것처럼 내보내지 않습니다.

**구현할 처리**

1. `review["passed"]`가 True이면 `END`로 보내며 `status`를 `"통과"`로 적습니다. 이 검사는 라운드 상한 검사보다 먼저입니다. 마지막 라운드에 통과한 초안도 통과입니다.
2. 통과하지 못했고 `round`가 `max_rounds` 이상이면 `END`로 보내며 `status`를 `"라운드 상한 도달: 사람 확인 필요"`로 적습니다. 라운드는 더 올리지 않습니다.
3. 그 밖에는 `issue`가 `"research"`이면 `"planner"`, 아니면 `"writer"`로 보냅니다.
4. `round`를 1 올리고 `feedback`에 검수 의견(`review["feedback"]`)을 담아 `Command`로 반환합니다. 입력 State는 바꾸지 않습니다.

**반환값: `Command`**

| 인자 | 값 |
|---|---|
| `goto` | 통과나 상한이면 `END`, 아니면 `"planner"` 또는 `"writer"` |
| `update` | 종료: `{"status": 문자열}`, 되돌림: `{"round": 현재 값 + 1, "feedback": 검수 의견}` |

**반환 예시:** `round`가 0이고 판정이 `{"passed": False, "issue": "research", "feedback": "경쟁사 조사가 없습니다."}`인 경우

```python
Command(goto="planner", update={"round": 1, "feedback": "경쟁사 조사가 없습니다."})
```

In [ ]:
# [노드] 다음 단계 결정
def supervisor(state: PTState) -> Command[Literal["planner", "writer", "__end__"]]:
    """검수 결과와 라운드 수로 끝낼지, 다시 쓸지, 다시 조사할지 정합니다."""
    # 1. 통과이면 END로 보내고 status를 "통과"로 적으세요.
    ...

    # 2. 라운드 상한에 닿았으면 END로 보내고 status를 "라운드 상한 도달: 사람 확인 필요"로 적으세요.
    ...

    # 3. issue가 research이면 planner, 아니면 writer로 보낼 곳을 정하세요.
    ...

    # 4. round를 1 올리고 검수 의견을 feedback에 담아 Command로 반환하세요.
    ...

In [ ]:
# [자가채점] 통과, 조사 부족, 글 수정, 상한 도달 네 경우의 이동과 갱신값을 확인합니다. 모델을 호출하지 않습니다.
def grade_question_6():
    base = make_input("평가용 요청", max_rounds=2)
    cases = {
        "passed": {"review": {"passed": True, "issue": "none", "feedback": ""}, "round": 1},
        "research": {"review": {"passed": False, "issue": "research", "feedback": "경쟁사 조사가 없습니다."}, "round": 0},
        "rewrite": {"review": {"passed": False, "issue": "rewrite", "feedback": "금액이 메모와 다릅니다."}, "round": 1},
        "limit": {"review": {"passed": False, "issue": "rewrite", "feedback": "금액이 메모와 다릅니다."}, "round": 2},
        "passed_at_limit": {"review": {"passed": True, "issue": "none", "feedback": ""}, "round": 2},
        "custom_limit": {"review": {"passed": False, "issue": "rewrite", "feedback": "금액이 메모와 다릅니다."}, "round": 2, "max_rounds": 3},
    }
    states = {}
    for name, values in cases.items():
        state = dict(base)
        state.update(values)
        states[name] = state
    before = deepcopy(states)

    done = supervisor(states["passed"])
    assert isinstance(done, Command), "Command를 반환하세요."
    assert done.goto == END and done.update["status"] == "통과", "통과면 END로 끝내고 status를 '통과'로 적으세요."

    again = supervisor(states["research"])
    assert again.goto == "planner", "조사가 부족하면 planner로 보냅니다."
    assert again.update == {"round": 1, "feedback": "경쟁사 조사가 없습니다."}, "round를 1 올리고 검수 의견을 feedback에 담으세요."

    rewrite = supervisor(states["rewrite"])
    assert rewrite.goto == "writer", "글 수정이면 writer로 보냅니다."
    assert rewrite.update == {"round": 2, "feedback": "금액이 메모와 다릅니다."}, "round를 1 올리고 검수 의견을 feedback에 담으세요."

    stop = supervisor(states["limit"])
    assert stop.goto == END, "상한에 닿으면 더 돌리지 말고 END로 끝내세요."
    assert stop.update["status"] == "라운드 상한 도달: 사람 확인 필요", "반려된 채 끝나면 status에 '라운드 상한 도달: 사람 확인 필요'를 적으세요."
    assert "round" not in stop.update, "상한에서는 라운드를 더 올리지 않습니다."
    last = supervisor(states["passed_at_limit"])
    assert last.goto == END and last.update["status"] == "통과", "통과 판정은 라운드 상한보다 먼저 확인하세요. 마지막 라운드에 통과한 초안도 통과입니다."

    custom = supervisor(states["custom_limit"])
    assert custom.goto == "writer", "상한은 2로 고정하지 말고 state의 max_rounds와 비교하세요."
    assert states == before, "입력 State를 직접 바꾸지 마세요."
    print("다음 단계 결정 확인 완료")


grade_question_6()

## 7. 그래프를 조립하고 두 요청을 실제로 실행하세요

지금까지 만든 노드를 연결하고 작은 요청과 큰 요청을 같은 그래프로 실행합니다. 모델은 이 문항의 실행 셀에서만 호출합니다.

| 제공 코드 | 입력과 반환값 |
|---|---|
| `collect(state)` | 병렬 조사가 모두 끝난 뒤 실행되어 `findings`를 한 글로 묶어 `{"brief": 문자열}`을 반환합니다. 요약하지 않고 묶기만 하며, 메모 대조는 reviewer가 맡습니다. |
| 실행 셀 | 두 요청을 `app.invoke`로 실행하고 `output/assignment02_runs.json`에 저장합니다. |

In [ ]:
# [노드] 결과 모으기: 병렬 조사가 모두 끝나면 실행되어 조사 메모를 한 글로 묶습니다.
def collect(state: PTState):
    """모든 조사 메모를 묶어 brief로 반환합니다."""
    return {"brief": format_findings(state["findings"])}

**작성:** 노드 사이의 연결을 완성하세요. 노드 등록과 컴파일, 그래프 그림 출력은 제공합니다.

**목적:** 계획, 병렬 조사, 결과 모으기, 작성, 검수가 이어지는 팀 그래프를 만듭니다.

**구현할 처리**

1. `planner` 뒤에 조건부 엣지로 라우팅 함수 `assign_research`를 연결합니다. 갈 수 있는 노드 목록은 `["researcher"]`입니다. `add_edge("planner", "researcher")`는 쓰지 않습니다.
2. `researcher` 뒤에는 `collect`를 고정 엣지로 연결합니다.
3. `collect` 뒤에는 `writer`를 고정 엣지로 연결합니다.
4. `reviewer` 뒤에는 `supervisor`를 고정 엣지로 연결합니다. `writer`와 `supervisor`에서 나가는 엣지는 만들지 않습니다. 도구와 `Command`가 다음 노드를 정합니다.

In [ ]:
# [그래프 연결] 노드 사이의 연결을 완성합니다.
builder = StateGraph(PTState)
builder.add_node("planner", planner)
builder.add_node("researcher", researcher)
builder.add_node("collect", collect)
# writer가 핸드오프로 갈 수 있는 곳(reviewer, supervisor)을 그래프 그림에 표시합니다. 실행에는 영향이 없습니다.
builder.add_node("writer", writer, destinations=("reviewer", "supervisor"))
builder.add_node("reviewer", reviewer)
builder.add_node("supervisor", supervisor)
builder.add_edge(START, "planner")

# 1. planner 뒤에 라우팅 함수 assign_research를 조건부 엣지로 연결하세요.
...

# 2. researcher 뒤에 collect를 연결하세요.
...

# 3. collect 뒤에 writer를 연결하세요.
...

# 4. reviewer 뒤에 supervisor를 연결하세요.
...
# supervisor에서 나가는 엣지는 만들지 않습니다. 반환하는 Command가 다음 노드를 정합니다.
app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
# [자가채점] 그래프 연결을 확인합니다. 모델을 호출하지 않습니다.
def grade_question_7_graph():
    pairs = []
    for edge in app.get_graph().edges:
        pairs.append((edge.source, edge.target))
    needed = [("planner", "researcher"), ("researcher", "collect"), ("collect", "writer"), ("reviewer", "supervisor")]
    for source, target in needed:
        assert (source, target) in pairs, f"{source} -> {target} 연결이 없습니다."
    assert "planner" in builder.branches, "planner 뒤는 add_conditional_edges(\"planner\", assign_research, [\"researcher\"])로 연결하세요. 고정 엣지로는 Send가 나가지 않습니다."
    assert ("planner", "researcher") not in builder.edges, "planner -> researcher 고정 엣지는 지우세요. Send가 연결을 만듭니다."
    for source, target in builder.edges:
        assert source not in ("writer", "supervisor"), f"{source}에서 나가는 고정 엣지({source} -> {target})는 만들지 마세요. 도구와 Command가 다음 노드를 정합니다."
    assert ("writer", "reviewer") in pairs and ("writer", "supervisor") in pairs, "제공된 writer 노드 등록의 destinations를 바꾸지 마세요."
    for source, target in pairs:
        if source == "reviewer":
            assert target == "supervisor", "reviewer에서는 supervisor로만 나갑니다."
    print("그래프 연결 확인 완료")


grade_question_7_graph()

In [ ]:
# 작은 요청과 큰 요청을 같은 그래프로 실제 실행합니다. 모델을 여러 번 부르므로 1~3분 걸립니다.
requests = {
    "small": "팀 주간회의에서 3분 동안 3분기 지역별 매출을 공유합니다. 지역별 매출 요약만 정리해 주세요.",
    "big": "다음 주 임원회의에서 중소기업용 구독 요금제 출시를 제안합니다. 발표는 10분, 슬라이드 8장 이내입니다. 3분기 고객군별 매출과 해지율, 중소기업 고객 설문, 경쟁사 요금제, 사내 PT 작성 원칙을 확인해서 발표 구성안과 3분 요약 스크립트를 써 주세요.",
}
runs = {}
for case_id, request in requests.items():
    final = app.invoke(make_input(request, max_rounds=2))
    # 작업 ID만 목록으로 뽑아 저장합니다.
    task_ids = []
    for task in final["tasks"]:
        task_ids.append(task["task_id"])
    runs[case_id] = {
        "request": request,
        "task_ids": task_ids,
        "findings": final["findings"],
        "draft": final["draft"],
        "review": final["review"],
        "round": final["round"],
        "status": final["status"],
    }
save_json("assignment02_runs.json", runs)
print("저장:", output_dir / "assignment02_runs.json")

In [ ]:
# 요청마다 작업 수가 달라졌는지, 어떻게 끝났는지 봅니다.
for case_id, run in runs.items():
    print(case_id, "| 마지막 계획 작업:", run["task_ids"], "| 조사 메모:", list(run["findings"]), "| 돌려보낸 횟수:", run["round"], "| 상태:", run["status"])

In [ ]:
# 큰 요청의 최종 발표 구성안입니다.
display(Markdown(runs["big"]["draft"]))

In [ ]:
# [자가채점] 실행 결과의 작업 수, 메모 키, 종료 상태, 저장 파일을 확인합니다. 모델을 추가로 호출하지 않습니다.
# [채점용 보조 함수] 발표 구성안의 쉼표가 든 숫자(예: 360,000)와 소수점이 있는 퍼센트 중 조사 메모에 그대로 없는 것을 찾습니다.
NUMBER_PATTERN = r"\d{1,3}(?:,\d{3})+|\d+\.\d+%"


def numbers_not_in_memo(draft, findings):
    """발표 구성안의 수치 중 조사 메모에 그대로 없는 것을 목록으로 반환합니다."""
    memo_text = format_findings(findings)

    missing = []
    for number in re.findall(NUMBER_PATTERN, draft):
        if number not in memo_text and number not in missing:
            missing.append(number)

    return missing


def grade_question_7_runs():
    assert set(runs) == {"small", "big"}, "두 요청의 결과가 모두 있어야 합니다."
    saved = json.loads((output_dir / "assignment02_runs.json").read_text(encoding="utf-8"))
    for case_id, run in runs.items():
        assert saved[case_id]["status"] == run["status"] and saved[case_id]["draft"] == run["draft"], "저장한 파일이 이번 실행 결과와 다릅니다."
        assert run["task_ids"], "작업이 하나도 만들어지지 않았습니다."
        for task_id in run["task_ids"]:
            assert task_id in run["findings"], f"작업 {task_id}의 조사 메모가 findings에 없습니다. researcher가 작업 ID를 키로 반환하는지 확인하세요."
        assert 0 <= run["round"] <= 2, "돌려보낸 횟수는 상한(2)을 넘으면 안 됩니다."
        if run["status"] == "통과":
            assert run["review"]["passed"] is True, "통과로 끝났다면 마지막 검수 결과도 통과여야 합니다."
            assert run["draft"].strip(), "통과인데 발표 구성안이 비어 있습니다. 저장 코드와 draft 필드를 확인하세요."
        else:
            assert run["status"] == "라운드 상한 도달: 사람 확인 필요", "통과가 아니면 status는 '라운드 상한 도달: 사람 확인 필요'여야 합니다."
            assert run["round"] == 2, "라운드 상한 도달로 끝났다면 round는 max_rounds(2)여야 합니다. 6번 supervisor가 state['max_rounds']와 비교하는지 확인하세요."
    assert len(runs["small"]["findings"]) < len(runs["big"]["findings"]), "작은 요청의 작업 수가 큰 요청보다 적어야 합니다."
    assert len(runs["big"]["findings"]) >= 3, "임원회의 요청은 매출·설문·경쟁사·작성 원칙 중 3개 이상을 조사해야 합니다."
    # 메모에서 그대로 찾지 못한 수치는 단위 변환일 수도, 모델이 계산한 값일 수도 있습니다.
    missing = numbers_not_in_memo(runs["big"]["draft"], runs["big"]["findings"])
    print("메모에서 그대로 찾지 못한 수치(8번에서 확인):", missing)
    print("실행 결과 확인 완료")


grade_question_7_runs()

## 8. 발표 구성안을 조사 메모와 대조해 발행 여부를 판단합니다

**배경:** 검수자가 통과시켜도 발표 구성안이 맞는다는 보장은 없습니다. 발행 전에 사람이 조사 메모와 직접 대조해야 합니다.

**작성:** 서술형 답안

**요구사항:**
- `runs["big"]["draft"]`에서 수치 두 개(매출이나 해지율 하나, 설문이나 요금 하나)를 골라 `runs["big"]["findings"]`의 메모와 대조하세요. 출처 ID가 붙어 있는지 적습니다. 7번 자가채점이 출력한 '메모에서 그대로 찾지 못한 수치' 목록이 있으면 단위 변환인지, 모델이 계산한 값인지 구분하세요.
- 슬라이드 수가 작성 원칙 G-2(8장 이내)를 지켰는지, 경쟁사 자료 중 확인일이 90일을 넘은 것(CP-C)을 어떻게 표시했는지 확인하세요. 메모에 CP-C가 없다면 그 이유(조사한 고객군)를 적으세요.
- 작은 요청과 큰 요청의 작업 수가 다른 이유를 `planner`가 읽은 요청 문장에서 찾아 설명하세요.
- `round`가 1 이상이면 다시 쓰기나 재조사가 있었습니다. 재조사가 있었다면 같은 작업 ID의 새 메모가 옛 메모를 대신하므로, 메모의 범위가 줄어들지 않았는지 확인하세요.
- `status`가 `통과`였는지, 몇 번 돌려보냈는지 적으세요. 이 구성안을 그대로 임원회의에 써도 되는지, 쓰기 전에 사람이 확인할 것은 무엇인지 판단하세요.

<details><summary>힌트</summary>

```text
접근방법: 발표 구성안 문장 하나와 그 근거 메모 한 줄을 나란히 놓고 값이 같은지 봅니다.
세부구현: 증감률과 해지율은 도구가 계산해 메모에 적어 둡니다. 구성안의 숫자가 메모 숫자와 같은지, 출처 ID와 슬라이드 수 원칙을 지켰는지만 보면 됩니다.
```

</details>

**확인:** 실제 결과를 근거로 설명하세요.

*(여기에 확인한 근거와 판단 이유를 작성하세요.)*